In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import PatchCollection
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import os
from pathlib import Path
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from minisom import MiniSom

In [2]:
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

In [3]:
# ------------------------- Basisverzeichnis (aktuelles Notebook-Verzeichnis) -------------------------
base_dir = Path.cwd()

# ------------------------- Suche nach dem neusten Preprocessing-Ordner -------------------------
preprocessing_root = base_dir.parent.parent / "3.1_Preprocessing" / "Preprocessing"
timestamp_folders = [f for f in preprocessing_root.iterdir() if f.is_dir()]
if not timestamp_folders:
    raise FileNotFoundError(f"Keine Preprocessing-Ordner in {preprocessing_root} gefunden.")

latest_folder = max(timestamp_folders, key=lambda f: f.stat().st_mtime)
print(f"Verwendeter Preprocessing-Ordner: {latest_folder.name}")

# ------------------------- Zeitstempel finden (neuesten Ordner) -------------------------
try:
    prep_ts = datetime.strptime(latest_folder.name, "%Y-%m-%d_%H-%M-%S")
except ValueError:
    print("Warnung: Preprocessing-Ordnername entspricht nicht dem Schema. Nutze Dateisystem-Zeit.")
    prep_ts = datetime.fromtimestamp(latest_folder.stat().st_mtime)

# ------------------------- Lade Preprocessed Data -------------------------
input_path_prep = latest_folder / "Preprocessed_SOM_Ready.csv"
df_full = pd.read_csv(input_path_prep, low_memory=False)

print(f"Daten erfolgreich aus 3.1_Preprocessing geladen! Shape: {df_full.shape}")

Verwendeter Preprocessing-Ordner: 2026-01-05_00-04-27


Daten erfolgreich aus 3.1_Preprocessing geladen! Shape: (94264, 24)


<div class="alert alert-info">
    <b>Temperature Analysis & Hexagonal SOM</b><br><br>
    <b>Datenquelle:</b><br>
    - Preprocessed Data: Ionen (Log + Gaussian Scaling) + Temperatur (Cleaned)<br>
    <br>
    <b>Filter:</b><br>
    - <b>Nur Temperaturen > 10 °C</b> werden berücksichtigt.
    <br>
    <b>Ziel:</b><br>
    - Clustering mittels SOM (Hexagonal).
    - Analyse der Temperaturverteilung innerhalb der Cluster.
    - Export als PDF Report.
</div>

In [4]:
# ------------------------- Features auswählen (nur Hauptionen) -------------------------
target_ions = ["Na", "Ca", "Mg", "Cl", "HCO3", "SO4"]
features = []
for col in df_full.columns:
    if "_gauss" in col:
        element = col.split("_")[0]
        if element in target_ions:
            features.append(col)

print(f"Input Features für SOM: {features}")

# ------------------------- Filtern -------------------------
df_som = df_full[features + ['temperature_in_c', 'rock_type']].copy()

# ------------------------- Temperatur in numerische Werte umwandeln -------------------------
df_som['temperature_in_c'] = pd.to_numeric(df_som['temperature_in_c'], errors='coerce')

# ------------------------- Nur Temperaturen über 10°C zulassen ODER fehlende Temperaturen (NaN) -------------------------
len_before_temp = len(df_som)
# Logik: (Temp >= 10) ODER (Temp ist NaN)
temp_condition = (df_som['temperature_in_c'] >= 10) | (df_som['temperature_in_c'].isna())
df_som = df_som[temp_condition]

print(f"Filter Temp >= 10°C or NaN: {len_before_temp - len(df_som)} Zeilen entfernt. Verbleibend: {len(df_som)}")

initial_len = len(df_som)
df_som.dropna(subset=features, inplace=True)
print(f"Zeilen mit fehlenden Features entfernt: {initial_len - len(df_som)}. Final: {len(df_som)}")

Input Features für SOM: ['Na_in_mmol/L_log_gauss', 'Mg_in_mmol/L_log_gauss', 'Ca_in_mmol/L_log_gauss', 'Cl_in_mmol/L_log_gauss', 'SO4_in_mmol/L_log_gauss', 'HCO3_in_mmol/L_log_gauss']
Filter Temp >= 10°C or NaN: 0 Zeilen entfernt. Verbleibend: 94264
Zeilen mit fehlenden Features entfernt: 19248. Final: 75016


In [5]:
# ------------------------- MinMax Scaling (nur für SOM Training Data) -------------------------
scaler = MinMaxScaler(feature_range=(0, 1))
data_values = df_som[features].values
data_scaled = scaler.fit_transform(data_values)

print("Daten skaliert.")

Daten skaliert.


In [6]:
# ------------------------- SOM Training -------------------------
som_x = 6 
som_y = 6
input_len = len(features)
sigma = 0.5
learning_rate = 0.5

som = MiniSom(x=som_x, y=som_y, input_len=input_len, 
              sigma=sigma, learning_rate=learning_rate, 
              topology='hexagonal', neighborhood_function='gaussian', 
              activation_distance='euclidean', random_seed=42)

som.random_weights_init(data_scaled)

print(f"Starte SOM Training ({som_x}x{som_y} Hexagonal)...")
som.train_random(data_scaled, 10000) 
print("Training beendet.")


Starte SOM Training (6x6 Hexagonal)...


Training beendet.


In [7]:
# ------------------------- Temperatur Analyse Vorbereitung -------------------------
winner_coords = []
for x in data_scaled:
    w = som.winner(x)
    winner_coords.append(w)

df_som['som_x'] = [c[0] for c in winner_coords]
df_som['som_y'] = [c[1] for c in winner_coords]

# ------------------------- Mean Temperature Map berechnen -------------------------
temp_map = np.full((som_x, som_y), np.nan)

agg = df_som.groupby(['som_x', 'som_y'])['temperature_in_c'].mean()

for (gx, gy), val in agg.items():
    temp_map[gx, gy] = val

print("Mean Temperature Map berechnet.")

Mean Temperature Map berechnet.


In [8]:
# ------------------------- Plot Funktion für PDF -------------------------
def plot_hex_map_to_axis(ax, data_matrix, title, cmap='viridis', label_suffix='', annotate=False):
    sy, sx = data_matrix.shape
    ax.set_aspect('equal')
    patches = []
    colors = []
    
    for y in range(sy):
        for x in range(sx):
            val = data_matrix[y, x]
            if np.isnan(val): 
                continue 
                
            offset = 0.5 if y % 2 != 0 else 0.0
            center_x = x + offset
            center_y = y * (np.sqrt(3) / 2)
            radius = 1 / np.sqrt(3)
            
            hex_poly = mpatches.RegularPolygon(
                (center_x, center_y), numVertices=6, radius=radius, 
                orientation=np.radians(30), edgecolor='k', linewidth=0.5
            )
            patches.append(hex_poly)
            colors.append(val)
            
            if annotate:
                ax.text(center_x, center_y, f"{val:.1f}", ha='center', va='center', 
                        fontsize=7, color='black', fontweight='bold')
            
    if not patches:
        return

    collection = PatchCollection(patches, cmap=cmap, alpha=0.9)
    collection.set_array(np.array(colors))
    ax.add_collection(collection)
    
    ax.set_xlim(-0.5, sx + 0.5)
    ax.set_ylim(-0.5, sy * (np.sqrt(3)/2) + 0.5)
    ax.axis('off')
    ax.set_title(title, fontsize=12)
    return collection

In [9]:
# ------------------------- PDF Report Erstellung -------------------------
output_root = base_dir / "MiniSom_Results"
output_root.mkdir(exist_ok=True)

# ------------------------- Create Run Folder -------------------------
current_ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
run_folder = output_root / f"Run_{current_ts}"
run_folder.mkdir(parents=True, exist_ok=True)
print(f"Output Folder: {run_folder}")


pdf_path = run_folder / "SOM_Report_Temperature.pdf"
print(f"Erstelle Report: {pdf_path} ...")

# ------------------------- Calculate Count -------------------------
temp_count = df_som['temperature_in_c'].count()
with PdfPages(pdf_path) as pdf:
    
    # ------------------------- Beschreibungsseite -------------------------
    plt.figure(figsize=(8, 6))
    plt.text(0.5, 0.5, f"SOM Temperature Analysis Report\n\nHexagonal Self-Organizing Map (10x10)\nInput: Major Ions (Log+Gauss)\nOverlay: Temperature (Mean & Dist)\nFILTER: Only Temp > 10\u00b0C\n\nTotal Data Points: {temp_count}", 
             ha='center', va='center', fontsize=20)
    plt.axis('off')
    pdf.savefig()
    plt.close()
    
    # ------------------------- Abstandsmatrix -------------------------
    fig, ax = plt.subplots(figsize=(8, 8))
    u_matrix = som.distance_map()
    col_u = plot_hex_map_to_axis(ax, u_matrix, "U-Matrix (Distance Map)", cmap='bone_r')
    plt.colorbar(col_u, ax=ax, fraction=0.046, pad=0.04, label='Distance')
    pdf.savefig(fig)
    plt.close(fig)
    
    # ------------------------- Durchschnittstemperatur -------------------------
    fig, ax = plt.subplots(figsize=(10, 10))
    col_t = plot_hex_map_to_axis(ax, temp_map, "Mean Temperature per Cluster (> 10°C)", cmap='coolwarm', annotate=True)
    plt.colorbar(col_t, ax=ax, fraction=0.046, pad=0.04, label='Temp °C')
    pdf.savefig(fig)
    plt.close(fig)
    
    # ------------------------- Component Plane -------------------------
    W = som.get_weights()
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    axes = axes.flatten()
    for i, feature in enumerate(features):
        short = feature.split("_")[0]
        col_c = plot_hex_map_to_axis(axes[i], W[:,:,i], f"{short}", cmap='viridis')
    plt.suptitle("Component Planes (Ion Distribution)")
    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)
    
    # ------------------------- Detaillierte Beschreibung -------------------------
    cluster_counts = df_som.groupby(['som_x', 'som_y']).size().sort_values(ascending=False)
    all_clusters = cluster_counts.index.tolist()
    
    plots_per_page = 9
    chunks = [all_clusters[i:i + plots_per_page] for i in range(0, len(all_clusters), plots_per_page)]
    
    print(f"Erstelle Verteilungs-Plots für {len(all_clusters)} Cluster auf {len(chunks)} Seiten...")
    
    for page_idx, chunk in enumerate(chunks):
        fig, axes = plt.subplots(3, 3, figsize=(12, 12))
        axes = axes.flatten()
        plt.suptitle(f"Temp Distribution (>10°C) - Page {page_idx + 1}/{len(chunks)} (Sorted by Size)", fontsize=16)
        
        for i, (cx, cy) in enumerate(chunk):
            subset = df_som[(df_som['som_x'] == cx) & (df_som['som_y'] == cy)]
            temps = subset['temperature_in_c'].dropna()
            
            ax = axes[i]
            if len(temps) == 0:
                ax.text(0.5, 0.5, "No Data", ha='center')
            else:
                sns.histplot(temps, kde=True, ax=ax, color='crimson', kde_kws={'cut': 0})
            
            count = len(temps)
            ax.set_title(f"Cluster ({cx}, {cy}) - N={count}")
            ax.set_xlabel("Temp °C")

        for j in range(len(chunk), plots_per_page):
            axes[j].axis('off')

        plt.tight_layout(rect=[0, 0.03, 1, 0.95]) 
        pdf.savefig(fig)
        plt.close(fig)

print("Report fertiggestellt.")

Output Folder: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.2_Machine-Learning\MiniSom\MiniSom_Results\Run_2026-01-05_00-05-16
Erstelle Report: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.2_Machine-Learning\MiniSom\MiniSom_Results\Run_2026-01-05_00-05-16\SOM_Report_Temperature.pdf ...


Erstelle Verteilungs-Plots für 36 Cluster auf 4 Seiten...


Report fertiggestellt.


In [10]:

# ------------------------- PDF Report Erstellung -------------------------
rock_pdf_path = output_root / "SOM_Report_RockType.pdf"
print(f"Erstelle Rock Type Report: {rock_pdf_path}...")

unique_rocks = df_som['rock_type'].dropna().unique()
rock_color_map = {
    'Sandstone': '#4B3F72',
    'Limestone': '#3A6B84',
    'Dolomite': '#2D898B',
    'Shale': '#419D78',
    'Sand and Gravel': '#419D78',
    'Anhydrite': '#82D173',
    'Metamorphic Rock': '#2C4F6B',
    'Chert': '#419D78',
    'Basalt': '#82D173',
    'Phyllite': '#419D78',
    'Siltstone': '#E0E0E0',
    'Coal': '#1F1F1F',
    'Empty': '#FFFFFF'
}
import matplotlib.colors as mcolors
base_colors = list(mcolors.TABLEAU_COLORS.values())
for i, r in enumerate(unique_rocks):
    if r not in rock_color_map:
        rock_color_map[r] = base_colors[i % len(base_colors)]

# ------------------------- Berechnung der dominanten Gesteinsart (Modus) -------------------------
# Rückkehr zur absoluten Mehrheit (Modus)
node_rock_types = {}
rock_map = np.full((som_x, som_y), 'Empty', dtype=object)

for x in range(som_x):
    for y in range(som_y):
        subset = df_som[(df_som['som_x'] == x) & (df_som['som_y'] == y)]
        if not subset.empty:
            rocks = subset['rock_type'].dropna().tolist()
            if rocks:
                node_rock_types[(x, y)] = rocks
                # Modus berechnen
                mode = max(set(rocks), key=rocks.count)
                rock_map[x, y] = mode

print("Dominant Rock Map calculated.")

from matplotlib.collections import PatchCollection
import matplotlib.patches as mpatches
from matplotlib.backends.backend_pdf import PdfPages

with PdfPages(rock_pdf_path) as pdf:
    # 1. Karte der Dominanten Gesteinsarten
    f, ax = plt.subplots(figsize=(10, 10))
    ax.set_aspect('equal')
    
    unique_map_rocks = set()
    
    for y in range(som_y):
        for x in range(som_x):
            val = rock_map[y, x]
            c = rock_color_map.get(val, '#FFFFFF')
            if val != 'Empty':
                unique_map_rocks.add(val)
            
            offset = 0.5 if y % 2 != 0 else 0.0
            center_x = x + offset
            center_y = y * (np.sqrt(3) / 2)
            radius = 1 / np.sqrt(3)
            
            hex_poly = mpatches.RegularPolygon(
                (center_x, center_y), numVertices=6, radius=radius, 
                orientation=np.radians(30), facecolor=c, edgecolor='k', linewidth=0.5
            )
            ax.add_patch(hex_poly)
            
            if val != 'Empty':
                 label = val[:4]
                 ax.text(center_x, center_y, label, ha='center', va='center', fontsize=7, color='white', weight='bold')

    ax.set_xlim(-1, som_y)
    ax.set_ylim(-1, som_x * np.sqrt(3)/2 + 1)
    plt.axis('off')
    
    # Legende
    legend_elements = [plt.Line2D([0], [0], marker='h', color='w', label=r, 
                          markerfacecolor=c, markersize=15) for r, c in rock_color_map.items() if r != 'Empty' and (r in unique_map_rocks)]
    
    plt.legend(handles=legend_elements, loc='center left', bbox_to_anchor=(1, 0.5), title="Rock Type")
    plt.title(f"Dominant Rock Type ({som_x}x{som_y} SOM)\n(Most Frequent)", fontsize=16)
    
    pdf.savefig(f, bbox_inches='tight')
    plt.close(f)
    
    # 2. Histogramme (Relative Anreicherung)
    print("Erstelle Rock Type Enrichment Plots per Cluster...")
    cluster_counts_grp = df_som.groupby(['som_x', 'som_y']).size().sort_values(ascending=False)
    all_clusters = cluster_counts_grp.index.tolist()
    
    plots_per_page = 9
    chunks = [all_clusters[i:i + plots_per_page] for i in range(0, len(all_clusters), plots_per_page)]
    
    # Globale Verteilung für Enrichment Plots
    global_counts = df_som['rock_type'].value_counts(normalize=True) * 100

    for page_idx, chunk in enumerate(chunks):
        fig, axes = plt.subplots(3, 3, figsize=(15, 12))
        axes = axes.flatten()
        plt.suptitle(f"Relative Enrichment (Delta %) - Page {page_idx + 1}/{len(chunks)}", fontsize=16)
        
        for i, (cx, cy) in enumerate(chunk):
            ax = axes[i]
            
            subset = df_som[(df_som['som_x'] == cx) & (df_som['som_y'] == cy)]
            subset = subset.dropna(subset=['rock_type'])
            
            if len(subset) > 0:
                c_counts = subset['rock_type'].value_counts(normalize=True) * 100
                
                # Abgleich
                all_idx = global_counts.index.union(c_counts.index)
                c_aligned = c_counts.reindex(all_idx, fill_value=0)
                g_aligned = global_counts.reindex(all_idx, fill_value=0)
                
                # Delta (Cluster % - Global %)
                delta = (c_aligned - g_aligned)
                
                # Top 5
                top_dev = delta.sort_values(ascending=False).head(5)
                
                sns.barplot(x=top_dev.values, y=top_dev.index, ax=ax, palette='coolwarm')
                ax.set_title(f"Cluster ({cx}, {cy}) - N={len(subset)}")
                ax.set_xlabel("Deviation (%)")
                ax.axvline(0, color='k', linestyle='--', linewidth=0.8)
            else:
                ax.text(0.5, 0.5, "No Data", ha='center')
        
        for j in range(len(chunk), plots_per_page):
            axes[j].axis('off')
            
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        pdf.savefig(fig)
        plt.close(fig)

print(f"Fertig! Rock PDF gespeichert: {rock_pdf_path}")


Erstelle Rock Type Report: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.2_Machine-Learning\MiniSom\MiniSom_Results\SOM_Report_RockType.pdf...
Dominant Rock Map calculated.


Erstelle Rock Type Enrichment Plots per Cluster...


C:\Users\lucca\AppData\Local\Temp\ipykernel_15972\4212708462.py:128: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_dev.values, y=top_dev.index, ax=ax, palette='coolwarm')
C:\Users\lucca\AppData\Local\Temp\ipykernel_15972\4212708462.py:128: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_dev.values, y=top_dev.index, ax=ax, palette='coolwarm')
C:\Users\lucca\AppData\Local\Temp\ipykernel_15972\4212708462.py:128: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_dev.values, y=top_dev.index, ax=ax, palette='coolwarm')
C:\Users\lucca\A

C:\Users\lucca\AppData\Local\Temp\ipykernel_15972\4212708462.py:128: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_dev.values, y=top_dev.index, ax=ax, palette='coolwarm')
C:\Users\lucca\AppData\Local\Temp\ipykernel_15972\4212708462.py:128: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_dev.values, y=top_dev.index, ax=ax, palette='coolwarm')
C:\Users\lucca\AppData\Local\Temp\ipykernel_15972\4212708462.py:128: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_dev.values, y=top_dev.index, ax=ax, palette='coolwarm')
C:\Users\lucca\A

C:\Users\lucca\AppData\Local\Temp\ipykernel_15972\4212708462.py:128: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_dev.values, y=top_dev.index, ax=ax, palette='coolwarm')
C:\Users\lucca\AppData\Local\Temp\ipykernel_15972\4212708462.py:128: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_dev.values, y=top_dev.index, ax=ax, palette='coolwarm')
C:\Users\lucca\AppData\Local\Temp\ipykernel_15972\4212708462.py:128: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_dev.values, y=top_dev.index, ax=ax, palette='coolwarm')
C:\Users\lucca\A

...

C:\Users\lucca\AppData\Local\Temp\ipykernel_15972\4212708462.py:128: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_dev.values, y=top_dev.index, ax=ax, palette='coolwarm')
C:\Users\lucca\AppData\Local\Temp\ipykernel_15972\4212708462.py:128: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_dev.values, y=top_dev.index, ax=ax, palette='coolwarm')
C:\Users\lucca\AppData\Local\Temp\ipykernel_15972\4212708462.py:128: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_dev.values, y=top_dev.index, ax=ax, palette='coolwarm')
C:\Users\lucca\A

C:\Users\lucca\AppData\Local\Temp\ipykernel_15972\4212708462.py:128: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_dev.values, y=top_dev.index, ax=ax, palette='coolwarm')
C:\Users\lucca\AppData\Local\Temp\ipykernel_15972\4212708462.py:128: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_dev.values, y=top_dev.index, ax=ax, palette='coolwarm')
C:\Users\lucca\AppData\Local\Temp\ipykernel_15972\4212708462.py:128: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_dev.values, y=top_dev.index, ax=ax, palette='coolwarm')
C:\Users\lucca\A

C:\Users\lucca\AppData\Local\Temp\ipykernel_15972\4212708462.py:128: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_dev.values, y=top_dev.index, ax=ax, palette='coolwarm')
C:\Users\lucca\AppData\Local\Temp\ipykernel_15972\4212708462.py:128: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_dev.values, y=top_dev.index, ax=ax, palette='coolwarm')
C:\Users\lucca\AppData\Local\Temp\ipykernel_15972\4212708462.py:128: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_dev.values, y=top_dev.index, ax=ax, palette='coolwarm')
C:\Users\lucca\A

C:\Users\lucca\AppData\Local\Temp\ipykernel_15972\4212708462.py:128: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_dev.values, y=top_dev.index, ax=ax, palette='coolwarm')


Fertig! Rock PDF gespeichert: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.2_Machine-Learning\MiniSom\MiniSom_Results\SOM_Report_RockType.pdf
